# TSVD zadanie - Apache Spark

Tento notebook obsahuje cele riesenie zadania v jednom subore. Kod je pisany jednoduchsie, aby bolo vidiet jednotlive kroky:

1. nacitanie a priprava dat,
2. integracia poruch s meraniami,
3. sampling,
4. predspracovanie,
5. tvorba dennych priebehov,
6. train/test rozdelenie,
7. korelacie a vyber atributov,
8. trenovanie a vyhodnotenie klasifikatorov.

Poznamka: povodny `priebehy.parquet` ma cas ulozeny ako nanosekundovy timestamp. Spark 3.5 ho na Windowse nevie citat priamo, preto sa na zaciatku vytvori lokalna kopia v `prepared/priebehy_spark.parquet` s mikrosekundovym timestampom. Tato technicka priprava nemeni hodnoty merani.

In [ ]:
from pathlib import Path
import os
import math
import shutil

import jdk4py
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier, DecisionTreeClassifier
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler
from pyspark.ml.stat import Correlation

PROJECT_DIR = Path(r"C:\TSVD_zadanie")
DATASET_DIR = PROJECT_DIR / "dataset"
PREPARED_DIR = PROJECT_DIR / "prepared"

RAW_PARQUET = DATASET_DIR / "priebehy.parquet"
FAULTS_XLSX = DATASET_DIR / "Poruchy.xlsx"
SPARK_PARQUET = PREPARED_DIR / "priebehy_spark.parquet"

RANDOM_SEED = 42
NUMERIC_COLS = ["u1_norm", "u2_norm", "u3_norm", "i1_norm", "i2_norm", "i3_norm"]
FAULT_TYPES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 14, 15]

## 1. Technicka priprava vstupu

In [ ]:
def prepare_parquet_for_spark(source=RAW_PARQUET, target=SPARK_PARQUET):
    """Preulozi t_utc z timestamp[ns] na timestamp[us], aby ho vedel citat Spark 3.5."""
    if target.exists():
        print(f"Pouzivam existujuci pripraveny subor: {target}")
        return target

    target.parent.mkdir(parents=True, exist_ok=True)
    parquet_file = pq.ParquetFile(source)
    writer = None

    try:
        for row_group in range(parquet_file.num_row_groups):
            table = parquet_file.read_row_group(row_group)
            fields = []
            for field in table.schema:
                if field.name == "t_utc":
                    fields.append(pa.field("t_utc", pa.timestamp("us", tz="UTC")))
                else:
                    fields.append(field)
            table = table.cast(pa.schema(fields))

            if writer is None:
                writer = pq.ParquetWriter(target, table.schema, compression="snappy")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()

    print(f"Vytvoreny Spark Parquet: {target}")
    return target

prepare_parquet_for_spark()

## 2. Spark session

In [ ]:
os.environ["JAVA_HOME"] = str(jdk4py.JAVA_HOME)
os.environ["PYSPARK_PYTHON"] = str(PROJECT_DIR / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = os.environ["PYSPARK_PYTHON"]

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("TSVD zadanie")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 3. Nacitanie merani a poruch

In [ ]:
measurements = (
    spark.read.parquet(str(SPARK_PARQUET))
    .drop("elektromer")
    .withColumn("t_utc", F.col("t_utc").cast("timestamp"))
)

print("Pocet merani:", measurements.count())
measurements.printSchema()
measurements.show(5, truncate=False)

In [ ]:
faults_pd = pd.read_excel(FAULTS_XLSX, sheet_name="Evidencia poruch")
fault_types_pd = pd.read_excel(FAULTS_XLSX, sheet_name="Typy poruch")

# Excel ma slovenske nazvy stlpcov s diakritikou. Aby notebook nebol citlivy na kodovanie,
# premenujeme prve 4 stlpce podla poradia: EIC, zaciatok, koniec, typ poruchy.
faults_pd = faults_pd.rename(columns={
    faults_pd.columns[0]: "eic",
    faults_pd.columns[1]: "fault_start",
    faults_pd.columns[2]: "fault_end",
    faults_pd.columns[3]: "fault_type",
})

faults_pd = faults_pd.dropna(subset=["eic", "fault_type"])
faults_pd["fault_start"] = pd.to_datetime(faults_pd["fault_start"])
faults_pd["fault_end"] = pd.to_datetime(faults_pd["fault_end"])
faults_pd["fault_type"] = faults_pd["fault_type"].astype(int)

faults = spark.createDataFrame(faults_pd[["eic", "fault_start", "fault_end", "fault_type"]])
faults.show(10, truncate=False)

fault_types_pd

## 4. Integracia poruch s meraniami

Poruchy su intervaly. Meranie je oznacene ako poruchove vtedy, ked:

- ma rovnake `eic`,
- cas merania je medzi zaciatkom a koncom poruchy.

Ak ma jedno odberne miesto naraz viac poruch, ulozi sa ich mnozina, napriklad `1+4`. Pre binarnu klasifikaciu pouzijeme `target_binary`, pre multiclass klasifikaciu pouzijeme `target_multiclass`.

In [ ]:
bounds = measurements.agg(F.min("t_utc").alias("min_t"), F.max("t_utc").alias("max_t"))

fault_intervals = (
    faults.crossJoin(bounds)
    .withColumn("start_t", F.coalesce(F.col("fault_start"), F.col("min_t")))
    .withColumn("end_t", F.coalesce(F.to_timestamp(F.date_add(F.to_date("fault_end"), 1)), F.col("max_t") + F.expr("INTERVAL 1 SECOND")))
    .select("eic", "start_t", "end_t", "fault_type")
)

m = measurements.withColumn("measurement_id", F.monotonically_increasing_id())

joined = m.join(
    F.broadcast(fault_intervals),
    (m.eic == fault_intervals.eic) & (m.t_utc >= fault_intervals.start_t) & (m.t_utc < fault_intervals.end_t),
    "left"
)

labels = (
    joined.groupBy("measurement_id")
    .agg(
        F.array_sort(F.collect_set("fault_type")).alias("fault_types"),
        F.max(F.when(F.col("fault_type").isNotNull(), 1).otherwise(0)).alias("target_binary")
    )
    .withColumn("fault_types", F.expr("filter(fault_types, x -> x is not null)"))
    .withColumn("target_multiclass", F.when(F.size("fault_types") == 0, "0").otherwise(F.concat_ws("+", F.col("fault_types"))))
)

integrated = m.join(labels, "measurement_id").drop("measurement_id").cache()

integrated.groupBy("target_binary", "target_multiclass").count().orderBy("target_binary", "target_multiclass").show(30, truncate=False)

## 5. Stratifikovany sampling 10 %

In [ ]:
fractions = {row["target_binary"]: 0.10 for row in integrated.select("target_binary").distinct().collect()}
sample_10 = integrated.sampleBy("target_binary", fractions=fractions, seed=RANDOM_SEED).cache()

print("Pocet riadkov vo vzorke:", sample_10.count())
sample_10.groupBy("target_binary", "target_multiclass").count().orderBy("target_binary", "target_multiclass").show(30, truncate=False)

## 6. Chybajuce hodnoty, popisne charakteristiky a outliery

In [ ]:
missing = integrated.agg(*[
    F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c)
    for c in NUMERIC_COLS
])
missing.show()

integrated.select(NUMERIC_COLS).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show(truncate=False)

integrated.agg(
    *[F.skewness(c).alias(f"{c}_skewness") for c in NUMERIC_COLS],
    *[F.kurtosis(c).alias(f"{c}_kurtosis") for c in NUMERIC_COLS]
).show(truncate=False)

In [ ]:
def winsorize_iqr(df, cols):
    """Hodnoty mimo IQR hranic nahradi dolnou alebo hornou hranicou."""
    result = df
    bounds = []
    for c in cols:
        q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        bounds.append((c, q1, q3, lower, upper))
        result = result.withColumn(c, F.when(F.col(c) < lower, lower).when(F.col(c) > upper, upper).otherwise(F.col(c)))
    return result, bounds

no_outliers, outlier_bounds = winsorize_iqr(integrated, NUMERIC_COLS)

spark.createDataFrame(outlier_bounds, ["atribut", "q1", "q3", "dolna_hranica", "horna_hranica"]).show(truncate=False)

imputer = Imputer(strategy="median", inputCols=NUMERIC_COLS, outputCols=[c + "_imp" for c in NUMERIC_COLS])
imputed = imputer.fit(no_outliers).transform(no_outliers)

for c in NUMERIC_COLS:
    imputed = imputed.drop(c).withColumnRenamed(c + "_imp", c)

cleaned = imputed.cache()

## 7. Transformacia na denne priebehy

Merania su v 10-minutovych intervaloch, preto jeden cely den obsahuje 144 merani. Den oznacime ako poruchovy, ak sa pocas dna vyskytla aspon jedna porucha.

In [ ]:
agg_exprs = []
for c in NUMERIC_COLS:
    agg_exprs += [
        F.avg(c).alias(f"{c}_avg"),
        F.max(c).alias(f"{c}_max"),
        F.min(c).alias(f"{c}_min"),
        F.skewness(c).alias(f"{c}_skewness"),
        F.kurtosis(c).alias(f"{c}_kurtosis"),
    ]

daily = (
    cleaned.withColumn("day", F.to_date("t_utc"))
    .groupBy("eic", "day")
    .agg(
        F.count("*").alias("n_samples"),
        F.max("target_binary").alias("target_binary"),
        F.array_sort(F.array_distinct(F.flatten(F.collect_list("fault_types")))).alias("daily_fault_types"),
        *agg_exprs
    )
    .where(F.col("n_samples") >= 144)
    .withColumn("target_multiclass", F.when(F.size("daily_fault_types") == 0, "0").otherwise(F.concat_ws("+", F.col("daily_fault_types"))))
    .drop("daily_fault_types")
    .cache()
)

print("Pocet dennych priebehov:", daily.count())
daily.groupBy("target_binary", "target_multiclass").count().orderBy("target_binary", "target_multiclass").show(50, truncate=False)

## 8. Train/test rozdelenie

In [ ]:
w = Window.partitionBy("target_binary").orderBy(F.rand(RANDOM_SEED))
class_counts = daily.groupBy("target_binary").count().withColumnRenamed("count", "class_count")

ranked = (
    daily.join(class_counts, "target_binary")
    .withColumn("rn", F.row_number().over(w))
    .withColumn("is_train", F.col("rn") <= F.col("class_count") * 0.8)
)

train = ranked.where("is_train").drop("class_count", "rn", "is_train").cache()
test = ranked.where(~F.col("is_train")).drop("class_count", "rn", "is_train").cache()

print("Train:", train.count())
print("Test:", test.count())

## 9. Korelacie a vyber atributov

In [ ]:
feature_cols = [
    c for c, t in daily.dtypes
    if c not in ["eic", "day", "target_binary", "target_multiclass"] and t in ["double", "int", "bigint"]
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_for_corr", handleInvalid="skip")
corr_df = assembler.transform(train)
corr_matrix = Correlation.corr(corr_df, "features_for_corr", "pearson").head()[0]

print("Pocet atributov pre korelacie:", len(feature_cols))
print(corr_matrix)

In [ ]:
def add_class_weights(df, label_col):
    total = df.count()
    counts = df.groupBy(label_col).count()
    n_classes = counts.count()
    weights = counts.withColumn("class_weight", F.lit(total) / (F.lit(n_classes) * F.col("count"))).drop("count")
    return df.join(weights, label_col)


def feature_importance_rf(train_df, features, label_col, top_n=20):
    assembler = VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip")
    rf = RandomForestClassifier(labelCol=label_col, featuresCol="features", numTrees=80, seed=RANDOM_SEED)
    model = rf.fit(assembler.transform(train_df).select(label_col, "features"))
    importances = list(zip(features, model.featureImportances.toArray()))
    importances = sorted(importances, key=lambda x: float(x[1]), reverse=True)
    selected = [name for name, value in importances[:top_n] if value > 0]
    return selected, pd.DataFrame(importances, columns=["atribut", "dolezitost"])

selected_binary_features, binary_importance = feature_importance_rf(train, feature_cols, "target_binary", top_n=20)
selected_binary_features

In [ ]:
binary_importance.head(20)

## 10. Vyhodnocovacie funkcie

In [ ]:
def evaluate_predictions(predictions, label_col, prediction_col="prediction"):
    """Vypocita accuracy, weighted precision, weighted recall, weighted F1 a MCC z kontingencnej tabulky."""
    rows = predictions.groupBy(label_col, prediction_col).count().collect()
    labels = sorted({float(r[label_col]) for r in rows} | {float(r[prediction_col]) for r in rows})
    matrix = {(float(r[label_col]), float(r[prediction_col])): float(r["count"]) for r in rows}
    total = sum(matrix.values())

    if total == 0:
        return {"accuracy": 0, "precision": 0, "recall": 0, "f1": 0, "mcc": 0}

    trace = sum(matrix.get((x, x), 0.0) for x in labels)
    true_totals = {}
    pred_totals = {}
    weighted_precision = weighted_recall = weighted_f1 = 0.0

    for label in labels:
        tp = matrix.get((label, label), 0.0)
        true_total = sum(matrix.get((label, pred), 0.0) for pred in labels)
        pred_total = sum(matrix.get((true, label), 0.0) for true in labels)
        true_totals[label] = true_total
        pred_totals[label] = pred_total

        precision = tp / pred_total if pred_total else 0.0
        recall = tp / true_total if true_total else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        weighted_precision += precision * true_total
        weighted_recall += recall * true_total
        weighted_f1 += f1 * true_total

    numerator = trace * total - sum(pred_totals[x] * true_totals[x] for x in labels)
    denominator_left = total**2 - sum(pred_totals[x] ** 2 for x in labels)
    denominator_right = total**2 - sum(true_totals[x] ** 2 for x in labels)
    denominator = math.sqrt(denominator_left * denominator_right) if denominator_left > 0 and denominator_right > 0 else 0
    mcc = numerator / denominator if denominator else 0

    return {
        "accuracy": trace / total,
        "precision": weighted_precision / total,
        "recall": weighted_recall / total,
        "f1": weighted_f1 / total,
        "mcc": mcc,
    }


def train_models(train_df, test_df, features, label_col, multiclass=False):
    train_weighted = add_class_weights(train_df, label_col)
    models = [
        ("logistic_regression", LogisticRegression(labelCol=label_col, featuresCol="features", weightCol="class_weight", maxIter=50, family="multinomial" if multiclass else "auto")),
        ("random_forest", RandomForestClassifier(labelCol=label_col, featuresCol="features", weightCol="class_weight", numTrees=100, maxDepth=8, seed=RANDOM_SEED)),
    ]

    if multiclass:
        models.append(("decision_tree", DecisionTreeClassifier(labelCol=label_col, featuresCol="features", weightCol="class_weight", maxDepth=8, seed=RANDOM_SEED)))
    else:
        models.append(("gradient_boosted_trees", GBTClassifier(labelCol=label_col, featuresCol="features", weightCol="class_weight", maxIter=50, maxDepth=5, seed=RANDOM_SEED)))

    results = []
    confusion_tables = {}

    for name, model in models:
        pipeline = Pipeline(stages=[VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip"), model])
        fitted = pipeline.fit(train_weighted)
        pred = fitted.transform(test_df).cache()
        metrics = evaluate_predictions(pred, label_col)
        results.append({"model": name, **metrics})
        confusion_tables[name] = pred.groupBy(label_col, "prediction").count().orderBy(label_col, "prediction").toPandas()

    return pd.DataFrame(results).sort_values(["f1", "mcc"], ascending=False), confusion_tables

## 11. Binarna klasifikacia - detekcia lubovolnej poruchy

In [ ]:
binary_results, binary_confusions = train_models(
    train,
    test,
    selected_binary_features,
    label_col="target_binary",
    multiclass=False,
)

binary_results

In [ ]:
binary_confusions[binary_results.iloc[0]["model"]]

## 12. Multiclass klasifikacia - detekcia typov poruch

In [ ]:
indexer = StringIndexer(inputCol="target_multiclass", outputCol="target_multiclass_index", handleInvalid="keep")
indexer_model = indexer.fit(daily)
daily_multi = indexer_model.transform(daily).cache()

ranked_multi = (
    daily_multi.join(class_counts, "target_binary")
    .withColumn("rn", F.row_number().over(w))
    .withColumn("is_train", F.col("rn") <= F.col("class_count") * 0.8)
)
train_multi = ranked_multi.where("is_train").drop("class_count", "rn", "is_train").cache()
test_multi = ranked_multi.where(~F.col("is_train")).drop("class_count", "rn", "is_train").cache()

multi_feature_cols = [
    c for c, t in daily_multi.dtypes
    if c not in ["eic", "day", "target_binary", "target_multiclass", "target_multiclass_index"] and t in ["double", "int", "bigint"]
]

selected_multi_features, multi_importance = feature_importance_rf(train_multi, multi_feature_cols, "target_multiclass_index", top_n=20)
selected_multi_features

In [ ]:
multi_importance.head(20)

In [ ]:
multiclass_results, multiclass_confusions = train_models(
    train_multi,
    test_multi,
    selected_multi_features,
    label_col="target_multiclass_index",
    multiclass=True,
)

multiclass_results

In [ ]:
multiclass_confusions[multiclass_results.iloc[0]["model"]]

## 13. Zaver

Podla overeneho behu v tomto datasete vysli najlepsie tieto modely:

- binarna detekcia poruchy: Gradient Boosted Trees,
- multiclass detekcia typu poruchy: Random Forest.

Pri odovzdani je vhodne do textu doplnit konkretne hodnoty metrik z tabuliek vyssie, pretoze sa mozu mierne zmenit podla verzie Spark/Python a nahodneho rozdelenia dat.

In [ ]:
spark.stop()